In [ ]:
import sys
import os
import requests
import json
# Add the src directory to the Python path
sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient

client = AuthentaClient(
    base_url="https://platform.authenta.ai",
    client_id="",
    client_secret="",
)

video_results = {}

In [ ]:
video_dir = "../data_samples/face_live_video/"
for folder in os.listdir(video_dir):
    if not folder.startswith('.'):
        for video in os.listdir(video_dir + folder):
            if video.lower().endswith(('.mp4', '.mov')):
                video_path = os.path.join(video_dir, folder, video)
                try:
                    print(f"Processing: {video_path}")
                    media = client.face_intelligence(
                        path=video_path,
                        model_type="FI-1",
                        livenessCheck=True,
                    )
                    result = requests.get(media["resultURL"])
                    video_results[video_path] = {"path": video_path, "result": json.loads(result.text)}
                except Exception as e:
                    print(f"Error processing {video_path}: {e}")


In [ ]:
from curl_cffi import requests as cffi_requests

resp = cffi_requests.post(
    "https://platform.authenta.ai/api/media",
    json={"name": "test.mp4", "contentType": "video/mp4", "size": 1000, "modelType": "FI-1"},
    headers={
        "x-client-id": "YOUR_ID",
        "x-client-secret": "YOUR_SECRET",
        "Content-Type": "application/json",
    },
    impersonate="chrome",  # mimics Chrome's TLS fingerprint
    timeout=30
)
print(resp.status_code, resp.text[:200])

In [14]:
media = client.face_intelligence(
    path = "../data_samples/face_similiar/person_1/A.jpeg",
    reference_img_path = "../data_samples/face_similiar/person_1/B.jpeg",
    model_type="FI-1",
    faceSimilarityCheck=True
    
)


create_media payload: {'name': 'A.jpeg', 'contentType': 'image/jpeg', 'size': 120289, 'modelType': 'FI-1', 'metadata': {'isSingleFace': True, 'faceswapCheck': False, 'livenessCheck': False, 'faceSimilarityCheck': True}}
create_media raw: 201 '{"mid":"69b84ac004bb20af0e3b8e0d","name":"A.jpeg","type":"Image","status":"INITED","modelType":"FI-1","createdAt":"2026-03-16T18:24:00.264Z","uploadUrl":"https://authenta-storage-prod.s3.us-east-1.ama'


In [15]:

result = requests.get(media["resultURL"])
print("\n".join(f"{k}: {v}" for k,v in json.loads(result.text).items()))

resultType: face-intelligence
isDeepFake: null
isLiveness: null
isSimilar: True
similarityScore: 82.67


### Method reference

#### `upload_file(path: str, model_type: str) -> Dict[str, Any]`
- Two-step upload helper.
- Calls `POST /media` with `name`, `contentType`, `size`, `modelType` to obtain `mid` and `uploadUrl`, then `PUT` the file to S3 using that URL.
- Returns the media JSON from `POST /media` (includes at least `mid`, `status`, `modelType`, timestamps, and possibly URLs).

#### `wait_for_media(mid: str, interval: float = 5.0, timeout: float = 600.0) -> Dict[str, Any]`
- Polls `GET /media/{mid}` until `status` is one of `PROCESSED`, `FAILED`, or `ERROR`, or until `timeout` seconds have elapsed.
- Sleeps `interval` seconds between polls.
- Returns the final media JSON or raises `TimeoutError` if it times out.

#### `list_media(**params) -> Dict[str, Any]`
- Wraps `GET /media`
- Returns all media in Authenta

#### `process(path: str, model_type: str, interval: float = 5.0, timeout: float = 600.0) -> Dict[str, Any]`
- High-level convenience method.
- Internally calls `upload_file` to create and upload the media, then `wait_for_media` with the returned `mid`.[file:3]
- Returns the final processed media JSON (including flags like `fake`, confidence scores, and `resultURL` / `heatmapURL` if present).

#### `delete_media(mid: str) -> None`
- Wraps `DELETE /media/{mid}`
- Deletes the given media record 


In [ ]:
# 1. Upload and wait for result

media = client.process("samples/nano_img.png", model_type="AC-1")
print(media["status"], media.get("fake"), media.get("resultURL"))

In [ ]:
# 2. Upload only 

upload_meta = client.upload_file("samples/nano_img.png", model_type="AC-1")
print(upload_meta["mid"], upload_meta["status"])

In [ ]:
# 3. Process from known mid
final_media = client.wait_for_media(upload_meta["mid"])

In [ ]:
# 4. List all media (mid)

all_media = client.list_media()
print(all_media)

In [ ]:
# 5. Delete a media

client.delete_media(upload_meta["mid"])

### Save Heatmaps for Images/Videos 

In [ ]:
from authenta.visualization import save_heatmap

media = client.process(
    "data_samples/nano_img2.png",
    model_type="AC-1",
)

# For AC-1, pass an output image path.
save_heatmap(
    media=media,
    out_path="results/image_heatmap.jpg",
    model_type="AC-1",
)

<!-- ### Save Artefacts for Images/Videos -->

In [ ]:
# ## Video
# src_video = "data_samples/test_00000121.mp4"
# media = client.process(src_video, model_type="DF-1")

# artefacts = save_video_artefacts(
#     media,
#     src_video_path=src_video,
#     out_dir="results"
#       )

In [ ]:
## Image 
media = client.process(
    "data_samples/nano_img2.png",
    model_type="AC-1",
)   

save_heatmap(
    media=media,
    out_path="results/image_heatmap.jpg",
    model_type="AC-1",
)

In [ ]:
save_heatmap(media=media, out_path="results/image_heatmap.jpg", model_type="AC-1")

In [ ]:
media2 = client.process("data_samples/test_00000121.mp4", model_type="DF-1")

save_heatmap_video(media2, "results")


In [ ]:
save_bounding_box_video(media2, src_video_path="data_samples/test_00000121.mp4", out_video_path="results/analyzed_video.mp4")
